# Qwen3 continuous batching 中间变量分析

这个 Notebook 读取 `llama-qwen3-batched-trace` 保存的真实 NPY/TSV，而不是构造概念示例。它覆盖：

- 四路异长 prefill；
- 四路 decode；
- 四个旧请求 decode 时加入一个 20-token 新请求；
- 五路 decode；
- embedding、RoPE、Q/K/V、物理 KV cache、active KV、mask、attention、28 层 hidden、final RMSNorm 和 LM-head logits；
- 单 GGUF 与 decoder/head 两个标准 shard 的逐 Tensor 运行结果对比。

> GGUF shard 是权重文件边界，不是运行时 graph 边界。两个 shard 由 loader 合并成一个 `llama_model`，所以推理过程中不存在 decoder 文件向 head 文件复制 hidden state 的步骤。

## 0. 生成数据与启动 Notebook

在仓库根目录运行：

```bash
cmake --build build-staged-bench --target llama-qwen3-batched-trace -j 16

build-staged-bench/bin/llama-qwen3-batched-trace \
  -m qwen3-1.7b/staged/qwen3-1.7B-BF16-staged-00001-of-00002.gguf \
  -c 512 -b 512 -ub 512 -t 8 \
  --trace-dir logs/qwen3_llama_batched_trace_split_complete_v2
```

只需要 NumPy 就能运行大部分单元。读取 GGUF 权重并验证 embedding/LM head 需要仓库 `.venv` 中的 `gguf`：

```bash
uv pip install --python .venv/bin/python jupyterlab
.venv/bin/python -m jupyter lab examples/qwen3-batched-trace/qwen3-batched-trace-analysis.ipynb
```

In [2]:
import csv
import json
import math
import os
from pathlib import Path

import numpy as np

np.set_printoptions(precision=6, suppress=True, linewidth=140, edgeitems=4)

def find_repo_root(start):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'CMakeLists.txt').is_file() and (path / 'src').is_dir():
            return path
    raise RuntimeError('run this notebook from inside the llama.cpp workspace')

def workspace_path(value):
    path = Path(value)
    return path if path.is_absolute() else REPO_ROOT / path

REPO_ROOT = find_repo_root(Path.cwd())
split_default = REPO_ROOT / 'logs/qwen3_llama_batched_trace_split_complete_v2'
mono_default = REPO_ROOT / 'logs/qwen3_llama_batched_trace_mono_complete_v2'
TRACE_DIR = workspace_path(os.environ.get('QWEN3_TRACE_DIR', split_default))
MONO_TRACE_DIR = workspace_path(os.environ.get('QWEN3_MONO_TRACE_DIR', mono_default))
STAGED_MANIFEST = workspace_path(os.environ.get(
    'QWEN3_STAGED_MANIFEST',
    'qwen3-1.7b/staged/qwen3-1.7B-BF16-staged.manifest.json',
))
PHASES = ['00_prefill_4', '01_decode_4', '02_join_new_request', '03_decode_5']
PHASE_NAME = os.environ.get('QWEN3_TRACE_PHASE', '02_join_new_request')
assert TRACE_DIR.is_dir(), f'missing trace directory: {TRACE_DIR}'
assert PHASE_NAME in PHASES
print('repo root :', REPO_ROOT)
print('trace     :', TRACE_DIR)
print('mono trace:', MONO_TRACE_DIR, '(exists=' + str(MONO_TRACE_DIR.is_dir()) + ')')
print('phase     :', PHASE_NAME)
print('manifest  :', STAGED_MANIFEST)

repo root : /home/qwe/workspace/llama.cpp
trace     : /home/qwe/workspace/llama.cpp/logs/qwen3_llama_batched_trace_split_complete_v2
mono trace: /home/qwe/workspace/llama.cpp/logs/qwen3_llama_batched_trace_mono_complete_v2 (exists=True)
phase     : 02_join_new_request
manifest  : /home/qwe/workspace/llama.cpp/qwen3-1.7b/staged/qwen3-1.7B-BF16-staged.manifest.json


In [3]:
CHECKS = {}

def check(name, condition, detail=''):
    passed = bool(condition)
    CHECKS[name] = passed
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f' - {detail}' if detail else ''))
    if not passed:
        raise AssertionError(name)

def read_tsv(path):
    with Path(path).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle, delimiter='\t'))

def npy_path(role, phase=PHASE_NAME, root=TRACE_DIR):
    return Path(root) / phase / f'{role}.npy'

def load(role, phase=PHASE_NAME, root=TRACE_DIR):
    return np.load(npy_path(role, phase, root), mmap_mode='r', allow_pickle=False)

def vector(role, phase=PHASE_NAME, root=TRACE_DIR, dtype=np.int64):
    return np.asarray(load(role, phase, root)).reshape(-1, order='F').astype(dtype, copy=False)

def compact_ranges(values):
    values = [int(value) for value in values]
    if not values:
        return 'none'
    groups = []
    start = end = values[0]
    for value in values[1:]:
        if value == end + 1:
            end = value
        else:
            groups.append(str(start) if start == end else f'{start}..{end}')
            start = end = value
    groups.append(str(start) if start == end else f'{start}..{end}')
    return ','.join(groups)

manifest_rows = read_tsv(TRACE_DIR / 'manifest.tsv')
manifest_by_role = {(row['phase'], row['role']): row for row in manifest_rows}

def tensor_summary(role, phase=PHASE_NAME, sample_count=8):
    array = load(role, phase)
    flat = np.asarray(array).reshape(-1, order='F')
    finite = np.isfinite(flat)
    finite_values = flat[finite]
    source_type = manifest_by_role.get((phase, role), {}).get('type', str(array.dtype))
    stats = (
        f'shape={list(array.shape)} npy={array.dtype} ggml={source_type} '
        f'finite={int(finite.sum())} zero={int(np.count_nonzero(flat == 0))} '
        f'-inf={int(np.count_nonzero(np.isneginf(flat)))}'
    )
    if finite_values.size:
        stats += f' min={float(finite_values.min()):.6g} max={float(finite_values.max()):.6g} mean={float(finite_values.mean()):.6g}'
    print(f'{role:44s} {stats}')
    print('  sample:', flat[:sample_count])
    return array

print('helper functions ready; manifest rows =', len(manifest_rows))

helper functions ready; manifest rows = 208


## 1. 四阶段运行流程

`T` 是本轮 flat token 数；`O` 是需要 logits 的输出行数；`U` 是写入后所有 sequence 的逻辑 KV token 数；`C` 是 attention 实际读取的、按 256 padding 的 active KV span。

In [4]:
print(f"{'phase':24s} {'T':>5s} {'O':>5s} {'U':>5s} {'C':>5s} {'positions by sequence'}")
print('-' * 105)
for phase in PHASES:
    tokens = vector('batch_token', phase)
    positions = vector('batch_position', phase)
    seq_ids = vector('batch_seq_id', phase)
    outputs = vector('batch_output', phase)
    memory = read_tsv(TRACE_DIR / phase / 'memory.tsv')
    logical = sum(int(row['logical_tokens']) for row in memory)
    active_span = load('attention_mask_layer0', phase).shape[0]
    position_text = ' '.join(
        f"seq{seq}:{positions[seq_ids == seq][0]}..{positions[seq_ids == seq][-1]}"
        for seq in np.unique(seq_ids)
    )
    print(f'{phase:24s} {tokens.size:5d} {int(outputs.sum()):5d} {logical:5d} {active_span:5d} {position_text}')

phase                        T     O     U     C positions by sequence
---------------------------------------------------------------------------------------------------------
00_prefill_4               240     4   240   256 seq0:0..47 seq1:0..55 seq2:0..63 seq3:0..71
01_decode_4                  4     4   244   256 seq0:48..48 seq1:56..56 seq2:64..64 seq3:72..72
02_join_new_request         24     5   268   512 seq0:49..49 seq1:57..57 seq2:65..65 seq3:73..73 seq4:0..19
03_decode_5                  5     5   273   512 seq0:50..50 seq1:58..58 seq2:66..66 seq3:74..74 seq4:20..20


运行主线：

```text
flat llama_batch(token, pos, seq_id, output)
  -> KV slot allocation and cell metadata
  -> fresh attention mask from seq_id + position
  -> token embedding
  -> decoder layer 0..27
       RMSNorm -> Q/K/V -> Q/K RoPE -> KV write
       -> QK score -> mask -> softmax -> V aggregation
       -> Wo + residual -> FFN + residual
  -> last-layer output-row gather
  -> final RMSNorm
  -> LM-head matmul
  -> logits
```

## 2. `llama_batch`：不同请求只在 flat token 维拼接

下面打印 join 阶段。旧请求各贡献一个 decode token，新请求贡献 20 个 prefill token。position 仍然按 request 独立。

In [5]:
tokens = vector('batch_token')
positions = vector('batch_position')
seq_ids = vector('batch_seq_id')
outputs = vector('batch_output')

print(f"{'i':>3s} {'token':>8s} {'seq':>4s} {'pos':>5s} {'output':>7s}")
for index, (token, seq_id, position, output) in enumerate(zip(tokens, seq_ids, positions, outputs)):
    print(f'{index:3d} {token:8d} {seq_id:4d} {position:5d} {output:7d}')
output_rows = np.flatnonzero(outputs)
print('flat T       =', tokens.size)
print('output rows  =', output_rows.tolist())
print('output O     =', output_rows.size)
print('position     =', positions.tolist())

  i    token  seq   pos  output
  0       13    0    49       1
  1     2309    1    57       1
  2     1096    2    65       1
  3     5546    3    73       1
  4    14076    4     0       0
  5     3040    4     1       0
  6    28682    4     2       0
  7     2937    4     3       0
  8      323    4     4       0
  9     7822    4     5       0
 10     4627    4     6       0
 11       13    4     7       0
 12     1096    4     8       0
 13     1681    4     9       0
 14      702    4    10       0
 15     1181    4    11       0
 16     1828    4    12       0
 17     3840    4    13       0
 18      323    4    14       0
 19     2309    4    15       0
 20     5546    4    16       0
 21       13    4    17       0
 22    28871    4    18       0
 23     3040    4    19       1
flat T       = 24
output rows  = [0, 1, 2, 3, 23]
output O     = 5
position     = [49, 57, 65, 73, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


## 3. 单 GGUF 与两个 shard 的运行结果

两个 trace 使用同一 Release binary、相同线程数和相同 workload。这里逐文件比较所有公共 NPY。若全部相同，说明文件拆分没有改变 embedding、mask、KV、hidden 或 logits。

In [6]:
if MONO_TRACE_DIR.is_dir():
    compared = 0
    mismatches = []
    for phase in PHASES:
        split_files = {path.name: path for path in (TRACE_DIR / phase).glob('*.npy')}
        mono_files = {path.name: path for path in (MONO_TRACE_DIR / phase).glob('*.npy')}
        for name in sorted(split_files.keys() & mono_files.keys()):
            split_array = np.load(split_files[name], mmap_mode='r', allow_pickle=False)
            mono_array = np.load(mono_files[name], mmap_mode='r', allow_pickle=False)
            compared += 1
            if split_array.shape != mono_array.shape or not np.array_equal(split_array, mono_array, equal_nan=True):
                mismatches.append((phase, name))
    check('monolithic vs split: all common NPY are elementwise identical', not mismatches, f'compared={compared}')
else:
    print('SKIP: monolithic trace not found:', MONO_TRACE_DIR)

[PASS] monolithic vs split: all common NPY are elementwise identical - compared=224


## 4. Embedding、position 和 RoPE

Qwen3 没有把 learned position embedding 加到 token embedding。`position` 只作为 Q/K RoPE 的输入：

```text
X[:,t] = token_embd.weight[:, token[t]]
Q = RoPE(QHeadNorm(Wq * RMSNorm(X)), position)
K = RoPE(KHeadNorm(Wk * RMSNorm(X)), position)
```

In [7]:
for role in [
    'graph_input_tokens',
    'input_embedding_layer0_hidden',
    'attention_norm_hidden_layer0',
    'q_before_rope_layer0',
    'position_ids_graph',
    'q_after_rope_layer0',
    'k_before_rope_layer0',
    'k_after_rope_layer0',
    'v_current_flat_layer0',
]:
    tensor_summary(role)

graph_tokens = vector('graph_input_tokens')
graph_positions = vector('position_ids_graph')
check('batch token == graph input token', np.array_equal(tokens, graph_tokens))
check('batch position == graph RoPE position', np.array_equal(positions, graph_positions))

q_before = np.asarray(load('q_before_rope_layer0'))[:, :, :, 0]
q_after = np.asarray(load('q_after_rope_layer0'))[:, :, :, 0]
q_delta = np.abs(q_after - q_before)
print('Q RoPE delta: max=', float(q_delta.max()), 'mean=', float(q_delta.mean()))
for query in range(tokens.size):
    if query < 6 or outputs[query]:
        print(f'  q={query:2d} seq={seq_ids[query]} pos={positions[query]:2d} max_delta={q_delta[:, :, query].max():.7g}')
zero_position_queries = np.flatnonzero(positions == 0)
check('RoPE at position 0 leaves Q unchanged', np.array_equal(q_before[:, :, zero_position_queries], q_after[:, :, zero_position_queries]))

graph_input_tokens                           shape=[24, 1, 1, 1] npy=float32 ggml=i32 finite=24 zero=0 -inf=0 min=13 max=28871 mean=5038.08
  sample: [   13.  2309.  1096.  5546. 14076.  3040. 28682.  2937.]
input_embedding_layer0_hidden                shape=[2048, 24, 1, 1] npy=float32 ggml=f32 finite=49152 zero=0 -inf=0 min=-0.12793 max=0.125 mean=-0.000253068
  sample: [-0.015564  0.00066  -0.053711 -0.038574  0.030151 -0.027588  0.008301 -0.033203]
attention_norm_hidden_layer0                 shape=[2048, 24, 1, 1] npy=float32 ggml=f32 finite=49152 zero=0 -inf=0 min=-1.62309 max=1.99067 mean=-0.00222693
  sample: [-0.030061  0.001728 -0.124489 -0.079472  0.078943 -0.055653  0.020902 -0.071731]
q_before_rope_layer0                         shape=[128, 16, 24, 1] npy=float32 ggml=f32 finite=49152 zero=0 -inf=0 min=-24.4192 max=20.3454 mean=0.00111126
  sample: [ 0.204046 -0.01582   0.015222 -0.60881   0.08187  -0.14034   0.020449  0.148217]
position_ids_graph                          

## 5. KV slot、物理 cache 和 active view

在 unified KV 模式中，五个 sequence 共用一个容量为 512 cell 的物理 cache。当前 K/V 按 `kv_slot_indices` scatter 写入；attention 再创建长度为 `C` 的 active view。

In [8]:
def slot_owners_through(phase_name):
    owners = {}
    for phase in PHASES[:PHASES.index(phase_name) + 1]:
        for row in read_tsv(TRACE_DIR / phase / 'kv_writes.tsv'):
            owners[int(row['slot'])] = {
                'seq_id': int(row['seq_id']),
                'position': int(row['position']),
                'token': int(row['token']),
                'phase': phase,
            }
    return owners

owners = slot_owners_through(PHASE_NAME)
slots = vector('kv_slot_indices')
print('current write slots:', compact_ranges(slots))
for batch_index, slot in enumerate(slots):
    owner = owners[int(slot)]
    print(f"  batch[{batch_index:2d}] -> slot={slot:3d} seq={owner['seq_id']} pos={owner['position']:2d} token={owner['token']}")

k_current = np.asarray(load('k_after_rope_layer0'))[:, :, :, 0]
v_current = np.asarray(load('v_current_flat_layer0'))[:, :, 0, 0]
physical_k = np.asarray(load('physical_k_cache_after_write_layer0'))[:, :, 0, 0]
physical_v = np.asarray(load('physical_v_cache_after_write_layer0'))[:, :, 0, 0]
active_k = np.asarray(load('active_k_permuted_layer0'))[:, :, :, 0]
active_v = np.asarray(load('active_v_permuted_layer0'))[:, :, :, 0]
D, Hkv, T = k_current.shape
capacity = physical_k.shape[1]
C = active_k.shape[1]

expected_current_k = k_current.reshape(D * Hkv, T, order='F').astype(np.float16).astype(np.float32)
check('current K is written to physical slots after F16 conversion', np.array_equal(expected_current_k, physical_k[:, slots]))
expected_active_k = physical_k.reshape(D, Hkv, capacity, order='F').transpose(0, 2, 1)[:, :C, :]
check('active K is physical K reshape + permute', np.array_equal(expected_active_k, active_k))
expected_active_v = physical_v.reshape(-1, order='F').reshape(capacity, D, Hkv, order='F')[:C]
check('active V is the transposed physical V layout', np.array_equal(expected_active_v, active_v))
expected_current_v = v_current.reshape(D, Hkv, T, order='F').transpose(2, 0, 1).astype(np.float16).astype(np.float32)
check('current V is written to physical slots after F16 conversion', np.array_equal(expected_current_v, active_v[slots]))

memory = read_tsv(TRACE_DIR / PHASE_NAME / 'memory.tsv')
logical_used = sum(int(row['logical_tokens']) for row in memory)
print(f'K current={list(k_current.shape)}, physical K={list(physical_k.shape)}, active K={list(active_k.shape)}')
print(f'V current={list(v_current.shape)}, physical V={list(physical_v.shape)}, active V={list(active_v.shape)}')
print(f'logical used U={logical_used}, active span C={C}, physical capacity={capacity}')

current write slots: 244..267
  batch[ 0] -> slot=244 seq=0 pos=49 token=13
  batch[ 1] -> slot=245 seq=1 pos=57 token=2309
  batch[ 2] -> slot=246 seq=2 pos=65 token=1096
  batch[ 3] -> slot=247 seq=3 pos=73 token=5546
  batch[ 4] -> slot=248 seq=4 pos= 0 token=14076
  batch[ 5] -> slot=249 seq=4 pos= 1 token=3040
  batch[ 6] -> slot=250 seq=4 pos= 2 token=28682
  batch[ 7] -> slot=251 seq=4 pos= 3 token=2937
  batch[ 8] -> slot=252 seq=4 pos= 4 token=323
  batch[ 9] -> slot=253 seq=4 pos= 5 token=7822
  batch[10] -> slot=254 seq=4 pos= 6 token=4627
  batch[11] -> slot=255 seq=4 pos= 7 token=13
  batch[12] -> slot=256 seq=4 pos= 8 token=1096
  batch[13] -> slot=257 seq=4 pos= 9 token=1681
  batch[14] -> slot=258 seq=4 pos=10 token=702
  batch[15] -> slot=259 seq=4 pos=11 token=1181
  batch[16] -> slot=260 seq=4 pos=12 token=1828
  batch[17] -> slot=261 seq=4 pos=13 token=3840
  batch[18] -> slot=262 seq=4 pos=14 token=323
  batch[19] -> slot=263 seq=4 pos=15 token=2309
  batch[20] -> 

## 6. Attention mask 不是多个 request mask 的 `CONCAT`

每个 ubatch 都根据 flat query 和 KV cell metadata 重新生成：

```text
M[c,t] = 0
  if cell[c] is occupied
     and cell[c].seq_id == query[t].seq_id
     and cell[c].position <= query[t].position

M[c,t] = -inf otherwise
```

旧 `[256,4]` mask 不会被 pad 后与新 request mask 拼接；join 阶段直接 fresh fill `[512,24]`。

In [9]:
mask = np.asarray(load('attention_mask_layer0'))[:, :, 0, 0]
actual_visible = np.isfinite(mask)
expected_visible = np.zeros((C, T), dtype=bool)
for slot, owner in owners.items():
    if slot < C:
        expected_visible[slot] = (seq_ids == owner['seq_id']) & (positions >= owner['position'])

check('mask finite pattern == reconstructed seq_id/position rule', np.array_equal(actual_visible, expected_visible))
check('visible mask values are exactly 0', np.all(mask[actual_visible] == 0))
check('blocked mask values are exactly -inf', np.all(np.isneginf(mask[~actual_visible])))

visible_counts = actual_visible.sum(axis=0)
print('per-query visible KV count:')
for query in range(T):
    visible_slots = np.flatnonzero(actual_visible[:, query])
    print(f'  q={query:2d} seq={seq_ids[query]} pos={positions[query]:2d} count={visible_counts[query]:3d} slots={compact_ranges(visible_slots)}')
print('finite zero =', int(actual_visible.sum()))
print('-inf       =', int((~actual_visible).sum()))

new_queries = np.flatnonzero(seq_ids == 4)
new_slots = [slot for slot, owner in owners.items() if owner['seq_id'] == 4 and slot < C]
print('\nseq4 causal triangle (# visible, . blocked):')
print('     query position:', ''.join(str(positions[q] // 10) if positions[q] >= 10 else ' ' for q in new_queries))
print('                     ', ''.join(str(positions[q] % 10) for q in new_queries))
for slot in new_slots:
    row = ''.join('#' if actual_visible[slot, query] else '.' for query in new_queries)
    print(f"slot {slot:3d} pos {owners[slot]['position']:2d}: {row}")

[PASS] mask finite pattern == reconstructed seq_id/position rule
[PASS] visible mask values are exactly 0
[PASS] blocked mask values are exactly -inf
per-query visible KV count:
  q= 0 seq=0 pos=49 count= 50 slots=0..47,240,244
  q= 1 seq=1 pos=57 count= 58 slots=48..103,241,245
  q= 2 seq=2 pos=65 count= 66 slots=104..167,242,246
  q= 3 seq=3 pos=73 count= 74 slots=168..239,243,247
  q= 4 seq=4 pos= 0 count=  1 slots=248
  q= 5 seq=4 pos= 1 count=  2 slots=248..249
  q= 6 seq=4 pos= 2 count=  3 slots=248..250
  q= 7 seq=4 pos= 3 count=  4 slots=248..251
  q= 8 seq=4 pos= 4 count=  5 slots=248..252
  q= 9 seq=4 pos= 5 count=  6 slots=248..253
  q=10 seq=4 pos= 6 count=  7 slots=248..254
  q=11 seq=4 pos= 7 count=  8 slots=248..255
  q=12 seq=4 pos= 8 count=  9 slots=248..256
  q=13 seq=4 pos= 9 count= 10 slots=248..257
  q=14 seq=4 pos=10 count= 11 slots=248..258
  q=15 seq=4 pos=11 count= 12 slots=248..259
  q=16 seq=4 pos=12 count= 13 slots=248..260
  q=17 seq=4 pos=13 count= 14 slot

## 7. GQA 多头注意力重算

Qwen3-1.7B 有 16 个 Q head 和 8 个 KV head，所以每两个 Q head 共用一个 K/V head：

```text
kv_head(h) = floor(h / 2)
S[c,t,h] = sum_d K[d,c,kv_head(h)] * Q[d,h,t]
P[:,t,h] = softmax(S[:,t,h] / sqrt(128) + M[:,t])
A[d,t,h] = sum_c V[c,d,kv_head(h)] * P[c,t,h]
```

In [10]:
scores = np.asarray(load('attention_scores_layer0'))[:, :, :, 0]
probabilities = np.asarray(load('attention_probabilities_layer0'))[:, :, :, 0]
context = np.asarray(load('attention_context_layer0'))[:, :, :, 0]
merged = np.asarray(load('attention_merged_heads_layer0'))[:, :, 0, 0]
Hq = q_after.shape[1]
group_size = Hq // Hkv

scores_rebuilt = np.empty_like(scores)
for head in range(Hq):
    scores_rebuilt[:, :, head] = active_k[:, :, head // group_size].T @ q_after[:, head, :]
score_error = float(np.max(np.abs(scores_rebuilt - scores)))
check('QK^T with GQA head mapping', np.allclose(scores_rebuilt, scores, atol=0.05, rtol=0.002), f'max_abs={score_error:.7g}')

masked_scores = scores / math.sqrt(D) + mask[:, :, None]
max_per_column = np.max(masked_scores, axis=0, keepdims=True)
exp_scores = np.exp(masked_scores - max_per_column)
exp_scores[~np.isfinite(masked_scores)] = 0
probabilities_rebuilt = exp_scores / exp_scores.sum(axis=0, keepdims=True)
prob_error = float(np.max(np.abs(probabilities_rebuilt - probabilities)))
check('scaled masked softmax', np.allclose(probabilities_rebuilt, probabilities, atol=1e-6, rtol=1e-6), f'max_abs={prob_error:.7g}')
check('masked attention probability is 0', np.max(np.where(~actual_visible[:, :, None], np.abs(probabilities), 0)) == 0)
prob_sum_error = float(np.max(np.abs(probabilities.sum(axis=0) - 1)))
check('attention probabilities sum to 1 over KV cells', prob_sum_error < 1e-6, f'max_abs={prob_sum_error:.7g}')

context_rebuilt = np.empty_like(context)
for head in range(Hq):
    context_rebuilt[:, :, head] = active_v[:, :, head // group_size].T @ probabilities[:, :, head]
context_error = float(np.max(np.abs(context_rebuilt - context)))
check('V aggregation with GQA head mapping', np.allclose(context_rebuilt, context, atol=4e-4, rtol=5e-4), f'max_abs={context_error:.7g}')

merged_rebuilt = context.transpose(0, 2, 1).reshape(D * Hq, T, order='F')
check('ConcatHeads layout', np.array_equal(merged_rebuilt, merged))

query = T - 1
head = 0
top_slots = np.argsort(probabilities[:, query, head])[-5:][::-1]
print(f'top attention slots for q={query}, seq={seq_ids[query]}, pos={positions[query]}, head={head}:')
for slot in top_slots:
    owner = owners.get(int(slot), {})
    print(f"  slot={slot:3d} p={probabilities[slot, query, head]:.7f} seq={owner.get('seq_id')} pos={owner.get('position')}")

[PASS] QK^T with GQA head mapping - max_abs=0.0357666
[PASS] scaled masked softmax - max_abs=4.768372e-07
[PASS] masked attention probability is 0
[PASS] attention probabilities sum to 1 over KV cells - max_abs=1.192093e-07
[PASS] V aggregation with GQA head mapping - max_abs=0.0002899468
[PASS] ConcatHeads layout
top attention slots for q=23, seq=4, pos=19, head=0:
  slot=267 p=0.1782145 seq=4 pos=19
  slot=262 p=0.1674171 seq=4 pos=14
  slot=265 p=0.1429166 seq=4 pos=17
  slot=258 p=0.0972005 seq=4 pos=10
  slot=252 p=0.0835586 seq=4 pos=4


## 8. Decoder hidden、output gather、final norm 和 logits

Layer 0 到 26 的 hidden 保持 `[H,T]`。最后一层 attention 仍对全部 `T` 个 token 计算并写 KV，然后根据 `batch.output` gather `O` 行；layer 27 FFN、final norm 和 LM head 因而是 `[H,O]` / `[V,O]`。

`attention_merged_heads_layer0` 是 `Wo` 之前的 ConcatHeads。当前 trace 没有单独保存 `Wo * ConcatHeads`，因此不能错误地断言 `post_attention == embedding + merged_heads`。

In [11]:
post_attention = np.asarray(load('post_attention_hidden_layer0'))
ffn_output = np.asarray(load('ffn_output_layer0'))
layer0 = np.asarray(load('decoder_output_hidden_layer0'))
check('layer0 FFN residual: post_attention + ffn_output == layer0', np.array_equal(post_attention + ffn_output, layer0))

print(f"{'layer':>5s} {'shape':>22s} {'rms':>12s} {'min':>12s} {'max':>12s}")
for layer in range(28):
    role = f'decoder_output_hidden_layer{layer}'
    path = npy_path(role)
    if not path.exists():
        print(f'{layer:5d} missing')
        continue
    hidden = np.asarray(load(role))
    rms = float(np.sqrt(np.mean(hidden * hidden)))
    print(f'{layer:5d} {str(list(hidden.shape)):>22s} {rms:12.6f} {hidden.min():12.6f} {hidden.max():12.6f}')

last_hidden = np.asarray(load('decoder_output_hidden_layer27'))
final_norm = np.asarray(load('final_norm_hidden'))
logits = np.asarray(load('lm_head_logits'))
check('last hidden token dimension == number of output rows', last_hidden.shape[1] == output_rows.size)
check('final norm token dimension == number of output rows', final_norm.shape[1] == output_rows.size)
check('logits token dimension == number of output rows', logits.shape[1] == output_rows.size)
print('last hidden:', list(last_hidden.shape))
print('final norm :', list(final_norm.shape))
print('logits     :', list(logits.shape))

logits_2d = logits[:, :, 0, 0]
for output_column, batch_index in enumerate(output_rows):
    top_ids = np.argsort(logits_2d[:, output_column])[-5:][::-1]
    values = logits_2d[top_ids, output_column]
    print(f'output_col={output_column} batch_i={batch_index} seq={seq_ids[batch_index]} pos={positions[batch_index]} top5={list(zip(top_ids.tolist(), values.tolist()))}')

[PASS] layer0 FFN residual: post_attention + ffn_output == layer0
layer                  shape          rms          min          max
    0       [2048, 24, 1, 1]     0.384189    -6.103717    12.422835
    1       [2048, 24, 1, 1]     0.899222   -49.566856    28.356066
    2       [2048, 24, 1, 1]    68.789925  -155.420227 12906.144531
    3       [2048, 24, 1, 1]    68.791862  -154.915009 12907.413086
    4       [2048, 24, 1, 1]    68.793068  -154.308182 12907.879883
    5       [2048, 24, 1, 1]    68.798668  -152.562073 12909.476562
    6       [2048, 24, 1, 1]    68.802750  -151.741608 12912.655273
    7       [2048, 24, 1, 1]    68.801979  -150.445801 12916.006836
    8       [2048, 24, 1, 1]    68.805672  -149.229462 12918.901367
    9       [2048, 24, 1, 1]    68.808022  -147.815460 12926.848633
   10       [2048, 24, 1, 1]    68.826752  -144.198120 12931.001953
   11       [2048, 24, 1, 1]    68.842377  -142.725098 12936.416016
   12       [2048, 24, 1, 1]    68.853485  -140.10

## 9. 直接读取 decoder/head shard 权重验证边界

这一节 mmap 两个 GGUF：

- 从 decoder shard 的 `token_embd.weight` 重做 embedding lookup；
- 从 head shard 的 `output_norm.weight` 重做 final RMSNorm；
- 只读取少量 `output.weight` 词表行，重做对应 logits。

BF16 权重只在选中行转换为 FP32，不会复制整个 622 MB 矩阵。

In [12]:
with STAGED_MANIFEST.open(encoding='utf-8') as handle:
    staged = json.load(handle)
print('architecture:', staged['architecture'])
print('total tensor:', staged['tensor_count'], 'bytes:', staged['tensor_bytes'])
for shard in staged['shards']:
    print(f"  split.no={shard['split_no']} role={shard['role']:7s} tensors={shard['tensor_count']:3d} bytes={shard['tensor_bytes']} path={shard['path']}")
    if shard['role'] == 'head':
        print('    head tensors:', [tensor['name'] for tensor in shard['tensors']])

try:
    from gguf import GGUFReader
except ImportError:
    print('SKIP GGUF numeric checks: run this notebook with .venv/bin/python after installing jupyterlab')
else:
    staged_dir = STAGED_MANIFEST.parent
    decoder_info = next(shard for shard in staged['shards'] if shard['role'] == 'decoder')
    head_info = next(shard for shard in staged['shards'] if shard['role'] == 'head')
    decoder_reader = GGUFReader(staged_dir / decoder_info['path'])
    head_reader = GGUFReader(staged_dir / head_info['path'])

    def gguf_tensor(reader, name):
        return next(tensor for tensor in reader.tensors if tensor.name == name)

    def bf16_rows(tensor, row_ids):
        raw = np.ascontiguousarray(tensor.data[np.asarray(row_ids, dtype=np.int64)])
        words = raw.view('<u2').reshape(len(row_ids), -1)
        return (words.astype(np.uint32) << 16).view('<f4').reshape(len(row_ids), -1)

    embedding_weight = gguf_tensor(decoder_reader, 'token_embd.weight')
    embedding_rows = bf16_rows(embedding_weight, tokens)
    runtime_embedding = np.asarray(load('input_embedding_layer0_hidden'))[:, :, 0, 0]
    check('decoder shard embedding lookup == runtime embedding', np.array_equal(embedding_rows.T, runtime_embedding))

    norm_weight = np.asarray(gguf_tensor(head_reader, 'output_norm.weight').data)
    epsilon = float(decoder_reader.fields['qwen3.attention.layer_norm_rms_epsilon'].contents())
    last_2d = last_hidden[:, :, 0, 0]
    norm_rebuilt = last_2d / np.sqrt(np.mean(last_2d * last_2d, axis=0, keepdims=True) + epsilon) * norm_weight[:, None]
    norm_error = float(np.max(np.abs(norm_rebuilt - final_norm[:, :, 0, 0])))
    check('head shard output RMSNorm == runtime final norm', np.allclose(norm_rebuilt, final_norm[:, :, 0, 0], atol=1e-5, rtol=1e-6), f'max_abs={norm_error:.7g}')

    top_ids = np.unique(np.argmax(logits_2d, axis=0))
    output_weight = gguf_tensor(head_reader, 'output.weight')
    selected_weight = bf16_rows(output_weight, top_ids)
    selected_logits = selected_weight @ final_norm[:, :, 0, 0]
    runtime_selected_logits = logits_2d[top_ids]
    logits_error = float(np.max(np.abs(selected_logits - runtime_selected_logits)))
    check('selected LM-head rows reproduce runtime logits', np.allclose(selected_logits, runtime_selected_logits, atol=0.02, rtol=0.002), f'rows={top_ids.tolist()} max_abs={logits_error:.7g}')

architecture: qwen3
total tensor: 311 bytes: 4063727616
  split.no=0 role=decoder tensors=309 bytes=3441389568 path=qwen3-1.7B-BF16-staged-00001-of-00002.gguf
  split.no=1 role=head    tensors=  2 bytes=622338048 path=qwen3-1.7B-BF16-staged-00002-of-00002.gguf
    head tensors: ['output_norm.weight', 'output.weight']
[PASS] decoder shard embedding lookup == runtime embedding
[PASS] head shard output RMSNorm == runtime final norm - max_abs=3.814697e-06
[PASS] selected LM-head rows reproduce runtime logits - rows=[13, 1096, 1681, 5546, 28682] max_abs=0.007604599


## 10. 对应源码调用链

```text
qwen3-batched-trace.cpp: begin_phase()
  -> llama_decode()                              src/llama-context.cpp
  -> llama_batch_allocr::split_simple()          src/llama-batch.cpp
  -> llama_kv_cache::find_slot()/apply_ubatch()  src/llama-kv-cache.cpp
  -> set_input_k_idxs()/set_input_kq_mask()      src/llama-kv-cache.cpp
  -> llama_model_qwen3::graph::graph()           src/models/qwen3.cpp
       -> build_inp_embd()/build_inp_pos()       src/llama-graph.cpp
       -> build_qkv()/RoPE
       -> cpy_k()/cpy_v()                        src/llama-kv-cache.cpp
       -> build_attn_mha()                       src/llama-graph.cpp
       -> residual + FFN
       -> output-row GET_ROWS
       -> final RMSNorm + output matmul
  -> ggml backend eval callback
  -> NPY/TSV files
```

关键位置：

- `examples/qwen3-batched-trace/qwen3-batched-trace.cpp`: workload、捕获点和 NPY 保存；
- `include/llama.h`: `llama_batch` 定义；
- `src/llama-kv-cache.cpp`: slot、K/V view 和 mask；
- `src/llama-graph.cpp`: embedding、attention 和 KV 写入；
- `src/models/qwen3.cpp`: Qwen3 decoder、最后一层 gather、final norm 和 LM head；
- `src/llama-model-loader.cpp`: 两个 GGUF shard 合并到一个 `weights_map`。

In [13]:
print('\nFinal validation summary')
print('-' * 80)
for name, passed in CHECKS.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
print(f'passed {sum(CHECKS.values())}/{len(CHECKS)} checks')
check('all executed checks passed', all(CHECKS.values()))


Final validation summary
--------------------------------------------------------------------------------
[PASS] monolithic vs split: all common NPY are elementwise identical
[PASS] batch token == graph input token
[PASS] batch position == graph RoPE position
[PASS] RoPE at position 0 leaves Q unchanged
[PASS] current K is written to physical slots after F16 conversion
[PASS] active K is physical K reshape + permute
[PASS] active V is the transposed physical V layout
[PASS] current V is written to physical slots after F16 conversion
[PASS] mask finite pattern == reconstructed seq_id/position rule
[PASS] visible mask values are exactly 0
[PASS] blocked mask values are exactly -inf
[PASS] QK^T with GQA head mapping
[PASS] scaled masked softmax
[PASS] masked attention probability is 0
[PASS] attention probabilities sum to 1 over KV cells
[PASS] V aggregation with GQA head mapping
[PASS] ConcatHeads layout
[PASS] layer0 FFN residual: post_attention + ffn_output == layer0
[PASS] last hidde